# 🚀 LLM Lifecycle: From Pre-Training to Post-Training (SFT & DPO)

Welcome! This notebook walks you through the entire lifecycle of a Large Language Model (LLM) on a **free Google Colab T4 GPU**.

We will:
1. **Load a Pre-trained Base Model (GPT-2)** and see how it behaves (Autocomplete only).
2. **Perform Supervised Fine-Tuning (SFT)** on a custom instruction dataset to teach it conversational form.
3. **Perform Direct Preference Optimization (DPO)** to align it with specific human preferences.
4. **Compare results** of the model at each stage.

--- 
*Note: Make sure your Colab Runtime is set to **GPU** (Runtime -> Change runtime type -> T4 GPU).*

## 📦 Step 1: Install Dependencies
We will install Hugging Face's `transformers` for loading models, `datasets` for handling data, and `trl` (Transformer Reinforcement Learning) which contains the SFT and DPO training loops.

In [ ]:
!pip install -q transformers datasets trl peft accelerate

## 🔍 Step 2: The Pre-trained Base Model (Autocomplete)
We'll load standard GPT-2 (124 Million parameters) as our base pre-trained model. We will ask it an instruction and observe that it behaves like an autocomplete engine, not a chatbot.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(model_id)

# Test prompt
prompt = "Question: How can I format a date in Python?\nAnswer:"
inputs = tokenizer(prompt, return_tensors="pt")

outputs = base_model.generate(**inputs, max_new_tokens=40, pad_token_id=tokenizer.eos_token_id)
print("--- BASE MODEL OUTPUT ---")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## ✍️ Step 3: Supervised Fine-Tuning (SFT)
Now, let's create a tiny SFT dataset containing instructions and matching answers. We will train the model to output a structured response when asked a question.

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

# Tiny SFT dataset mapping instructions to desired responses
sft_data = [
    {"text": "Question: How can I format a date in Python?\nAnswer: You can use datetime.now().strftime('%Y-%m-%d') to format dates."},
    {"text": "Question: How to write a file in Python?\nAnswer: Use open('file.txt', 'w') with a context manager."},
    {"text": "Question: What is list comprehension?\nAnswer: A concise way to create lists: [x for x in range(10)]."},
    {"text": "Question: How do you declare a variable in Python?\nAnswer: Simply assign a value: x = 5."},
    {"text": "Question: What does range do in Python?\nAnswer: It generates a sequence of numbers from start to stop."},
    {"text": "Question: How to check file size in Python?\nAnswer: Use os.path.getsize(path) from the built-in os module."}
]

sft_dataset = Dataset.from_list(sft_data)

# In newer versions of TRL, arguments like dataset_text_field and max_seq_length
# are passed inside SFTConfig (which inherits from TrainingArguments)
sft_config = SFTConfig(
    output_dir="./sft_outputs",
    per_device_train_batch_size=2,
    num_train_epochs=12,  # Overfit on this tiny dataset quickly to see behavior
    logging_steps=2,
    learning_rate=5e-5,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    report_to="none",
    dataset_text_field="text",
    max_length=128,
)

trainer = SFTTrainer(
    model=base_model,
    train_dataset=sft_dataset,
    args=sft_config,
)

print("--- Starting SFT Training ---")
trainer.train()
print("SFT Training Completed!")

### 🔍 Test the SFT Model
Let's see if the model has learned the conversational prompt/answer structure.

In [ ]:
inputs = tokenizer("Question: How can I format a date in Python?\nAnswer:", return_tensors="pt").to(base_model.device)
outputs = base_model.generate(**inputs, max_new_tokens=40, pad_token_id=tokenizer.eos_token_id)
print("--- SFT MODEL OUTPUT ---")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## ⚖️ Step 4: Direct Preference Optimization (DPO)
Now, let's say we want our model to be extremely concise. In our DPO preference dataset:
* The **chosen** (winning) response is highly direct and short.
* The **rejected** (losing) response is wordy and contains unnecessary text (like "Sure, here is how you do it!").

We will run DPO using `DPOTrainer` to steer the model towards direct answers.

In [ ]:
from trl import DPOTrainer, DPOConfig
import copy

# DPO Dataset structure: prompt, chosen response, and rejected response
dpo_data = [
    {
        "prompt": "Question: How can I format a date in Python?\nAnswer:",
        "chosen": "Use datetime.now().strftime('%Y-%m-%d').",
        "rejected": "Sure, here is the script. You can use datetime.now().strftime('%Y-%m-%d') to format dates. Hope this helps!"
    },
    {
        "prompt": "Question: How to write a file in Python?\nAnswer:",
        "chosen": "Use open('file.txt', 'w') with a context manager.",
        "rejected": "Hi! To write a file in Python, you need to use the open() function: open('file.txt', 'w'). Make sure to close it!"
    },
    {
        "prompt": "Question: What is list comprehension?\nAnswer:",
        "chosen": "A concise way to construct lists: [x for x in range(10)].",
        "rejected": "List comprehension is a really cool syntax. You write it like: [x for x in range(10)]. It makes code shorter."
    }
]

dpo_dataset = Dataset.from_list(dpo_data)

# We need a reference model. In DPO, we compare our active model against a copy of the SFT model
ref_model = copy.deepcopy(base_model)

# DPO configurations must be defined in DPOConfig
dpo_config = DPOConfig(
    output_dir="./dpo_outputs",
    per_device_train_batch_size=1,
    num_train_epochs=10,
    logging_steps=1,
    learning_rate=1e-5,
    fp16=torch.cuda.is_available(),
    report_to="none",
    max_length=256,
)

dpo_trainer = DPOTrainer(
    model=base_model,
    ref_model=ref_model,
    args=dpo_config,
    train_dataset=dpo_dataset,
    processing_class=tokenizer,
)

print("--- Starting DPO Training ---")
dpo_trainer.train()
print("DPO Training Completed!")

## 🏁 Step 5: Final Evaluation
Let's test the final aligned model on the same instruction and see if it outputs the direct, concise answer preferred in DPO.

In [ ]:
inputs = tokenizer("Question: How can I format a date in Python?\nAnswer:", return_tensors="pt").to(base_model.device)
outputs = base_model.generate(**inputs, max_new_tokens=40, pad_token_id=tokenizer.eos_token_id)
print("--- DPO ALIGNED MODEL OUTPUT ---")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))